<a href="https://colab.research.google.com/github/BiagioLuc/Visual-Place-Recognition-Project/blob/main/Progetto_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone --recursive https://github.com/BiagioLuc/Visual-Place-Recognition-Project.git

In [ ]:
%cd /content/Visual-Place-Recognition-Project/image-matching-models/
!pip install -e .[all]


In [ ]:
!pip install faiss-cpu

In [ ]:
%cd /content/Visual-Place-Recognition-Project/
!python download_datasets.py

In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--queries_folder '<path-to-queries-folder>' \
--num_preds_to_save 20 \
--recall_values 1 5 10 20 \
--save_for_uncertainty

In [ ]:
!python match_queries_preds.py \
--preds-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds' \
--matcher 'loftr' \
--device 'cuda' \
--num-preds 20

In [ ]:
!python reranking.py \
--preds-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds' \
--inliers-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds_loftr' \
--num-preds 20 \
--recall-values 1 5 10 20

## Start extension part

In [ ]:
!python VPR-methods-evaluation/main2.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--save_descriptors

In [ ]:
import torch
import os
from pathlib import Path


all_descriptors = torch.load("/content/Visual-Place-Recognition-Project/logs/log_dir/2026-06-02_20-45-37/database_descriptors.torch")

num_esempi = all_descriptors.shape[0]

train_dir = Path("file_train")
val_dir = Path("file_val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

split_idx = int(num_esempi * 0.8) #Splitting the descriptors for traning and validation

train_feats = all_descriptors[:split_idx]
val_feats = all_descriptors[split_idx:]

torch.save(train_feats, train_dir / "train_feats_part22.pt")
torch.save(val_feats, val_dir / "val_feats_part22.pt")

print(f"Train shape: {train_feats.shape} -> Saving in {train_dir}")
print(f"Val shape: {val_feats.shape} -> Saving in {val_dir}")


In [ ]:
!pip install accelerate circuitsvis datasets diffusers einops huggingface-hub nnsight pandas plotly scikit-learn sentencepiece transformers wandb umap-learn

In [ ]:
!git clone --recursive https://github.com/BiagioLuc/sae-for-vlm.git

In [ ]:
!pip install wandb
!wandb login

In [ ]:
!python sae-for-vlm/sae_train.py \
  --sae_model top_k \
  --activations_dir "/content/file_train" \
  --val_activations_dir "/content/file_val" \
  --device cuda:0 \
  --lr 1e-3 \
  --batch_size 1024 \
  --expansion_factor 2 \
  --steps 11000 \
  --decay_start 5000 \
  --k 128 \
  --seed 222

In [ ]:
!python sae-for-vlm/save_activations2.py \
  --vpr_descriptors_dir "/content/file_val" \
  --sae_checkpoint "/content/file_train_top_k_128_x2/trainer_0/ae.pt" \
  --output_dir "/content/activationsMixVPR" \
  --k 128 \
  --batch_size 1024

In [ ]:
!python sae-for-vlm/similarity_baseline \
--model "dinov2-base"\
--output_subdir "embeddings"\
--data_path "/content/dataset_split/val"


In [ ]:
!python sae-for-vlm/metric.py \
  --embeddings_path "/content/embeddings/embeddings_dinov2-base.pt" \
  --activations_dir "/content/activations_Cosplace" \
  --output_subdir "risultatiMetrica" \
  --device cuda:0

In [ ]:
import torch
import numpy as np
import os

# Script to find the 30 neuron indices, relatively at the top 10, medium 10, bottom 10 and their respctively 16 top images (activations)

#Load the scores
path_scores = "/content/activationsMixVPR/risultatiMetricaMixVPR/all_neurons_scores.pth"
monosemanticity = torch.load(path_scores, map_location='cpu')

#Exclude the neurons with values 0
valid_indices = torch.nonzero(~torch.isnan(monosemanticity)).squeeze()
valid_monosemanticity = monosemanticity[valid_indices]

#Sotoing of the vscores
sorted_values, sorted_indices_in_valid = torch.sort(valid_monosemanticity)


# top_10
top_10 = valid_indices[sorted_indices_in_valid[-10:]].flip(dims=[0]).tolist()

# mid_10
mid_point = len(sorted_indices_in_valid) // 2
mid_10 = valid_indices[sorted_indices_in_valid[mid_point - 5 : mid_point + 5]].tolist()

# bottom_10
bottom_10 = valid_indices[sorted_indices_in_valid[:10]].tolist()


selected_neurons = top_10 + mid_10 + bottom_10

print(f"-> Neurons Group 0 (Top 10 più monosemantici): {top_10}")
print(f"-> Neurons Group 1 (Mid 10 intermedi): {mid_10}")
print(f"-> Neurons Group 2 (Bottom 10 meno monosemantici): {bottom_10}")

#Load theactivations
path_activations = "/content/activationsMixVPR/0.pt"
print(f"\nCaricamento delle attivazioni da {path_activations}...")
activations = torch.load(path_activations, map_location='cpu')
print(f"Forma della matrice delle attivazioni: {activations.shape}")

# Extraction the top 16 images for the neurons selected
K = 16
hai_matrix = []

for neuron_id in selected_neurons:
    neuron_activations = activations[:, neuron_id]
    # Taken the values for each neuron, we select the top 16 values
    _, top_k_img_indices = torch.topk(neuron_activations, K, largest=True)
    hai_matrix.append(top_k_img_indices.numpy())

# Convert to matrix (Shape: 30, 16)
hai_matrix = np.array(hai_matrix)

#Saving the matrix
output_dir = "/content"
os.makedirs(output_dir, exist_ok=True)
output_npy_path = os.path.join(output_dir, "hai_matrix.npy")

np.save(output_npy_path, hai_matrix)
print(f"\n✓ File '{output_npy_path}' Successfully generated!")
print(f"Matrix Shape: {hai_matrix.shape} (30 neurons, 16 images)")

In [ ]:
import os
import glob
import zipfile
import math
from pathlib import Path

#Extraction of the images for validation and zipped the file for saving them
base_path = "/content/gsv_xs/train"
output_dir = Path("/content/immagini_val")
os.makedirs(output_dir, exist_ok=True)

cities = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
cities.sort()

print(f"Città processate in ordine alfabetico: {cities}\n")

for city in cities:
    city_path = os.path.join(base_path, city)

    images = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        images.extend(glob.glob(os.path.join(city_path, ext)))

    images.sort()

    totale_immagini_citta = len(images)
    if totale_immagini_citta == 0:
        continue

    num_da_prendere = math.ceil(totale_immagini_citta * 0.20)

    ultimo_20_immagini = images[-num_da_prendere:]

    city_zip_path = output_dir / f"{city}_images_part20.zip"
    print(f"Città: {city:<15} | Immagini totali: {totale_immagini_citta:<5} | Salvate nel file ZIP: {len(ultimo_20_immagini):<5}")

    with zipfile.ZipFile(city_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for img_path in ultimo_20_immagini:
            nome_file_puro = os.path.basename(img_path)
            zipf.write(img_path, arcname=nome_file_puro)

print(f"\nOperation completed. File ZIP saved in: {output_dir}")

In [ ]:
import os
import zipfile
import glob


cartella_zip = "/content/immagini_val"

estrazione_path = "/content/dataset_split/val"


file_zip_presenti = glob.glob(os.path.join(cartella_zip, "*.zip"))

for zip_path in file_zip_presenti:
    nome_file = os.path.basename(zip_path)
    nome_citta = nome_file.split("_")[0]
    cartella_citta_dest = os.path.join(estrazione_path, nome_citta)
    os.makedirs(cartella_citta_dest, exist_ok=True)


    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(cartella_citta_dest)
        print(f"Extraction city: {nome_citta} in {cartella_citta_dest}")

print("\nExtraction totally completed")

In [ ]:
!python sae-for-vlm/visualize_neurons.py \
--output_dir "Visualizzazione" \
--dataset_name "inat" \
--data_path "/content/dataset_split" \
--split "val" \
--group_fractions 0.34 0.34 0.32 \
--hai_indices_path  "/content/hai_matrix.npy"
